In [1]:
# !pip install pydicom
# !pip install pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg
!pip install --upgrade bitsandbytes
!pip install --upgrade transformers accelerate

^C
ERROR: Operation cancelled by user
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 87.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 61.1 MB/s eta 0:00:00:00:01
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.13.0
    Uninstalling accelerate-1.13.0:
      Successfully uninstalled accelerate-1.13.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transform

In [ ]:
"""
Knee abnormality report labeler — small model, batched, multi-GPU.

Same architecture as the original Qwen script (prompt -> JSON -> parse), but:
  * Qwen2.5-1.5B-Instruct instead of 7B  (~3GB fp16, fits trivially on one T4)
  * fp16, no quantization  (nf4 is SLOWER than fp16 on Turing; T4 has no bf16)
  * batched generation      (the actual 15-25x speedup)
  * chat template applied   (Instruct models degrade badly without it)
  * one full model replica per GPU, not device_map="auto" layer sharding

Kaggle 2x T4.
"""

import json
import logging
import os
import re
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
import pydicom
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # step up to -3B- if parse quality is poor
MODEL_NAME = "/kaggle/input/datasets/soumabhamajumdar2548/qwen25-1b5-instruct"
COMP_ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
BATCH_SIZE = 32          # drop to 16 if you see OOM with long reports
MAX_INPUT_TOKENS = 1536
MAX_NEW_TOKENS = 96      # a 12-key JSON is ~70 tokens; 256 was wasting decode steps
CHECKPOINT_EVERY = 5     # batches

LABEL_COLUMNS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

SYSTEM_PROMPT = (
    "You are a musculoskeletal radiologist extracting structured findings from "
    "knee MRI reports. You reply with a single JSON object and nothing else."
)


# --------------------------------------------------------------------------- #
# DICOM metadata (unchanged behaviour, trimmed fields)
# --------------------------------------------------------------------------- #

class DicomMeta:
    KEEP = ("Modality", "SeriesDescription", "BodyPartExamined", "Laterality")

    def __init__(self, base_path):
        self.base_path = base_path
        self.cache = {}

    def get(self, study_uid):
        if study_uid in self.cache:
            return self.cache[study_uid]

        study_path = Path(self.base_path) / str(study_uid)
        if not study_path.exists():
            self.cache[study_uid] = {}
            return {}

        dcm_files = list(study_path.rglob("*.dcm"))
        if not dcm_files:
            self.cache[study_uid] = {}
            return {}

        try:
            ds = pydicom.dcmread(dcm_files[0], stop_before_pixels=True)
            meta = {}
            for key in self.KEEP:
                val = ds.get(key, "")
                val = "" if val is None else str(val).strip()
                if val and val.upper() not in ("N/A", "NONE", "NULL"):
                    meta[key] = val
            self.cache[study_uid] = meta
            return meta
        except Exception as e:  # noqa: BLE001
            logging.warning(f"Error reading DICOM for {study_uid}: {e}")
            self.cache[study_uid] = {}
            return {}


# --------------------------------------------------------------------------- #
# One model replica pinned to one GPU
# --------------------------------------------------------------------------- #

class Replica:
    def __init__(self, model_name, device):
        self.device = device
        logging.info(f"Loading {model_name} onto {device}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
        ).to(device)
        self.model.eval()
        self.model.generation_config.pad_token_id = self.tokenizer.pad_token_id
        logging.info(f"Ready on {device}")

    @torch.inference_mode()
    def generate(self, prompts):
        texts = [
            self.tokenizer.apply_chat_template(
                [{"role": "system", "content": SYSTEM_PROMPT},
                 {"role": "user", "content": p}],
                tokenize=False,
                add_generation_prompt=True,
            )
            for p in prompts
        ]
        enc = self.tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_TOKENS,
        ).to(self.device)

        out = self.model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=self.tokenizer.pad_token_id,
        )
        gen = out[:, enc["input_ids"].shape[1]:]
        return self.tokenizer.batch_decode(gen, skip_special_tokens=True)


# --------------------------------------------------------------------------- #
# Labeler
# --------------------------------------------------------------------------- #

class KneeAbnormalityLabeler:
    def __init__(self, model_name=MODEL_NAME,
                 checkpoint_path="labeling_checkpoint.json",
                 dicom_base_path=f"{COMP_ROOT}/train_series/",
                 n_gpus=None):
        self.checkpoint_path = checkpoint_path
        self.dicom = DicomMeta(dicom_base_path)
        self.results = {}

        if os.path.exists(checkpoint_path):
            with open(checkpoint_path) as f:
                self.results = json.load(f)
            logging.info(f"Loaded checkpoint with {len(self.results)} completed rows")

        if n_gpus is None:
            n_gpus = max(1, torch.cuda.device_count())
        devices = [f"cuda:{i}" for i in range(n_gpus)] if torch.cuda.is_available() else ["cpu"]
        self.replicas = [Replica(model_name, d) for d in devices]
        self.pool = ThreadPoolExecutor(max_workers=len(self.replicas))
        logging.info(f"{len(self.replicas)} replica(s) live")

    # ---- prompt ---------------------------------------------------------- #

    def create_prompt(self, report_text, dicom_metadata=None):
        keys = ", ".join(f'"{c}"' for c in LABEL_COLUMNS)
        prompt = (
            "Read the knee MRI report below and decide, for each of 12 conditions, "
            "whether it is present.\n\n"
            f"Output a JSON object with exactly these keys: {keys}\n\n"
            "Values:\n"
            "  1  = condition is affirmatively described\n"
            "  0  = condition is explicitly negated or described as normal/intact\n"
            '  "?" = not mentioned, or the report is equivocal\n\n'
            "Do not infer a condition from an adjacent one. A meniscal tear does not "
            "imply an ACL tear. Respect laterality: medial findings go to medial keys "
            "only.\n\n"
        )
        if dicom_metadata:
            prompt += "Acquisition details:\n"
            for k, v in dicom_metadata.items():
                prompt += f"- {k}: {v}\n"
            prompt += "\n"
        prompt += f"REPORT:\n{report_text}\n\nJSON:"
        return prompt

    # ---- parsing (kept from original, slightly tightened) ---------------- #

    @staticmethod
    def parse_response(response_text):
        if not response_text:
            return None

        cleaned = re.sub(r"```(?:json)?\s*", "", response_text, flags=re.IGNORECASE)
        cleaned = re.sub(r"```\s*$", "", cleaned)

        start = cleaned.find("{")
        if start == -1:
            return None
        end = cleaned.rfind("}")

        if end != -1 and start < end:
            try:
                return json.loads(cleaned[start:end + 1])
            except json.JSONDecodeError:
                pass

        # Truncated output: close the object and drop any trailing partial pair.
        tail = cleaned[start:]
        tail = re.sub(r",\s*\"[^\"]*\"?\s*:?\s*[^,}]*$", "", tail)
        try:
            return json.loads(tail + "}")
        except json.JSONDecodeError:
            pass

        match = re.search(r"\{[^{}]*\}", cleaned)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                pass
        return None

    # ---- batch dispatch --------------------------------------------------- #

    def _run_batch(self, prompts):
        """Split one batch across replicas, run in parallel, reassemble in order."""
        n = len(self.replicas)
        if n == 1:
            return self.replicas[0].generate(prompts)

        chunks = [prompts[i::n] for i in range(n)]
        futures = [
            self.pool.submit(rep.generate, chunk)
            for rep, chunk in zip(self.replicas, chunks) if chunk
        ]
        outs = [f.result() for f in futures]

        merged = [None] * len(prompts)
        for i, out in enumerate(outs):
            merged[i::n] = out
        return merged

    def save_checkpoint(self):
        with open(self.checkpoint_path, "w") as f:
            json.dump(self.results, f)
        logging.info(f"Checkpoint saved: {len(self.results)} rows")

    # ---- main loop -------------------------------------------------------- #

    def label_dataframe(self, df):
        result_df = df.copy()
        if "inferred" not in result_df.columns:
            result_df["inferred"] = ""

        todo = []
        for idx in range(len(df)):
            if str(idx) in self.results:
                continue
            row = df.iloc[idx]
            if not any(pd.isna(row[c]) for c in LABEL_COLUMNS):
                continue
            if pd.isna(row["Report"]) or not str(row["Report"]).strip():
                self.results[str(idx)] = {"error": "empty_report"}
                continue
            todo.append(idx)

        logging.info(f"{len(todo)} rows need labeling")

        n_batches = (len(todo) + BATCH_SIZE - 1) // BATCH_SIZE
        parse_failures = 0

        for b in tqdm(range(n_batches), desc="Labeling"):
            batch_idx = todo[b * BATCH_SIZE:(b + 1) * BATCH_SIZE]
            prompts = []
            for idx in batch_idx:
                row = df.iloc[idx]
                meta = self.dicom.get(row["StudyInstanceUID"])
                prompts.append(self.create_prompt(str(row["Report"]), meta))

            responses = self._run_batch(prompts)

            for idx, resp in zip(batch_idx, responses):
                labels = self.parse_response(resp)
                if labels is None:
                    parse_failures += 1
                    self.results[str(idx)] = {"error": "parse_failed", "raw": resp[:200]}
                    continue

                inferred_cols = []
                for col in LABEL_COLUMNS:
                    if col not in labels:
                        continue
                    val = labels[col]
                    if val == "?" or val is None:
                        continue
                    try:
                        num = float(val)
                    except (ValueError, TypeError):
                        continue
                    if num in (0.0, 1.0):
                        result_df.at[idx, col] = int(num)
                        inferred_cols.append(col)

                if inferred_cols:
                    result_df.at[idx, "inferred"] = ",".join(inferred_cols)
                self.results[str(idx)] = {"completed": True, "inferred": inferred_cols}

            if (b + 1) % CHECKPOINT_EVERY == 0:
                self.save_checkpoint()

        self.save_checkpoint()
        logging.info(f"Parse failures: {parse_failures} / {len(todo)}")
        return result_df


def main():
    df = pd.read_csv(f"{COMP_ROOT}/train.csv")
    logging.info(f"Loaded {len(df)} rows; columns: {df.columns.tolist()}")

    labeler = KneeAbnormalityLabeler()
    labeled_df = labeler.label_dataframe(df)
    labeled_df.to_csv("train_labeled.csv", index=False)

    print("\n" + "=" * 50)
    print("LABELING SUMMARY")
    print("=" * 50)
    print(f"Total rows: {len(labeled_df)}")
    print(f"Rows with inferred labels: {labeled_df['inferred'].str.len().gt(0).sum()}")
    print("\nLabel distribution:")
    for col in LABEL_COLUMNS:
        non_nan = labeled_df[col].notna().sum()
        inferred = labeled_df["inferred"].str.contains(re.escape(col), na=False).sum()
        print(f"  {col:20s}: {non_nan:6d} total, {inferred:6d} inferred")


if __name__ == "__main__":
    main()

2026-09-06 22:33:11,657 - INFO - Loaded 4407 rows; columns: ['StudyInstanceUID', 'Report', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
2026-09-06 22:33:11,658 - INFO - Loading /kaggle/input/datasets/soumabhamajumdar2548/qwen25-1b5-instruct onto cuda:0...
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

2026-09-06 22:33:19,026 - INFO - Ready on cuda:0
2026-09-06 22:33:19,027 - INFO - Loading /kaggle/input/datasets/soumabhamajumdar2548/qwen25-1b5-instruct onto cuda:1...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

2026-09-06 22:33:22,553 - INFO - Ready on cuda:1
2026-09-06 22:33:22,554 - INFO - 2 replica(s) live
2026-09-06 22:33:22,744 - INFO - 4349 rows need labeling


Labeling:   0%|          | 0/136 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
2026-09-06 22:34:20,115 - INFO - Checkpoint saved: 160 rows
2026-09-06 22:35:26,612 - INFO - Checkpoint saved: 320 rows


# Slice-level feature extraction (RadImageNet / ImageNet ResNet50)

Extracts a fixed-length feature vector for **every DICOM slice** in the training set,
using a pretrained 2D CNN as a frozen encoder. Output is an `[N_slices, 2048]` matrix
plus a metadata CSV mapping each row back to `(StudyInstanceUID, SeriesInstanceUID, slice_index)`.

**Runs with internet OFF.** Nothing here downloads at run time. Pretrained weights are
resolved from attached Datasets / Kaggle Models / the torch hub cache; the torchvision
download is only attempted when cell 1's probe finds internet, and cell 2 refuses to
extract features from a randomly initialised backbone rather than wasting an hour on
noise. If you have no weights attached yet, run a dev session with internet on and call
`stage_weights_for_offline()` — cell 2's header comment spells out all three routes.

Cells:
1. Imports, config, offline preflight (deps, decoders, internet, data paths)
2. Backbone loader — offline-first weight resolution
3. DICOM preprocessing
4. Slice manifest + `Dataset`
5. Inference loop
6. Save features + metadata
7. Fine-tuning template (optional) + re-extraction

**Run cell 4 with `LIMIT_STUDIES = 20` first.** A full pass is hours of wall clock and
Kaggle sessions expire; the smoke test surfaces path and decoder problems in two minutes.

In [4]:
# =============================================================================
# CELL 1 - Imports, configuration, offline preflight
# =============================================================================
# Kaggle "Add dependencies" list (installed BEFORE the no-internet run starts,
# so this is the only pip that can happen). Paste into the notebook's
# environment/dependency settings, NOT into a code cell:
#
#     pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg python-gdcm
#
# torch / torchvision / numpy / pandas / opencv / tqdm ship with the Kaggle GPU
# image. The pylibjpeg + gdcm trio is what lets pydicom decode JPEG-compressed
# MRI; without it some series raise on .pixel_array and get skipped.
# -----------------------------------------------------------------------------

import gc
import json
import os
import sys
import time
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# OpenCV spawns threads per worker process; with 4 DataLoader workers that
# oversubscribes the 4 vCPUs and actually slows decoding down.
cv2.setNumThreads(0)

# --------------------------------------------------------------------------- #
# Configuration - everything tunable lives here
# --------------------------------------------------------------------------- #

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)    # dropout / head init, on both T4s

COMP_ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
WORK = Path("/kaggle/working")

IMG_SIZE = 224          # ResNet50 / ConvNeXt native input
BATCH_SIZE = 128        # per-step total, split across GPUs by DataParallel
NUM_WORKERS = min(4, os.cpu_count() or 2)   # Kaggle 2xT4 sessions give 4 vCPUs
FEATURE_DTYPE = np.float16                  # halves the on-disk feature matrix

# Smoke test first. A full pass over every slice is hours of wall clock and
# Kaggle sessions expire; 20 studies surfaces path/decoder problems in minutes.
# Set to None for the real run.
LIMIT_STUDIES = 20

# The 12 study-level targets (same order as the labeler cell above).
LABEL_COLUMNS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

# Output names
FEAT_PT = WORK / "slice_features_resnet50_radimagenet.pt"
META_CSV = WORK / "slice_metadata.csv"
FEAT_PT_FT = WORK / "slice_features_resnet50_finetuned.pt"
META_CSV_FT = WORK / "slice_metadata_finetuned.csv"
MANIFEST_CSV = WORK / "slice_manifest.csv"      # cached file listing

# --------------------------------------------------------------------------- #
# Helpers
# --------------------------------------------------------------------------- #

def seed_worker(worker_id):
    """Give each DataLoader worker its own numpy stream, derived from SEED.

    Forked workers inherit the parent's numpy RNG state, so without this all
    NUM_WORKERS of them draw the SAME slice jitter - and redraw it identically
    each epoch. torch already gives each worker base_seed + worker_id (and
    base_seed comes from the SEED-ed global RNG), so reusing it keeps numpy
    reproducible AND distinct per worker.
    """
    np.random.seed(torch.initial_seed() % 2**32)


def gpu_mem(tag=""):
    """Print per-GPU allocated/reserved memory. Call before and after big loads."""
    if not torch.cuda.is_available():
        print(f"[gpu_mem] {tag}: no CUDA")
        return
    parts = []
    for i in range(torch.cuda.device_count()):
        alloc = torch.cuda.memory_allocated(i) / 1024**3
        resv = torch.cuda.memory_reserved(i) / 1024**3
        total = torch.cuda.get_device_properties(i).total_memory / 1024**3
        parts.append(f"cuda:{i} {alloc:.2f}/{resv:.2f} of {total:.1f} GiB")
    print(f"[gpu_mem] {tag}: " + " | ".join(parts))


def resolve_comp_root(default=COMP_ROOT):
    """The competition mount point moved around between Kaggle UI versions."""
    candidates = [
        default,
        "/kaggle/input/rsna-knee-abnormality-detection",
        "/kaggle/input/competitions/rsna-knee-abnormality-detection",
    ]
    for c in candidates:
        if Path(c).exists():
            return c
    # Last resort: anything under /kaggle/input that holds a train.csv
    for p in sorted(Path("/kaggle/input").glob("*/train.csv")):
        return str(p.parent)
    raise FileNotFoundError(f"No competition data found. Tried: {candidates}")


# --------------------------------------------------------------------------- #
# Preflight
# --------------------------------------------------------------------------- #

print("=" * 70)
print("PREFLIGHT")
print("=" * 70)
print(f"python      : {sys.version.split()[0]}")
print(f"torch       : {torch.__version__}  (cuda {torch.version.cuda})")
print(f"torchvision : {torchvision.__version__}")
print(f"pydicom     : {pydicom.__version__}")
print(f"cuda avail  : {torch.cuda.is_available()}  n_gpu={torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i} -> {torch.cuda.get_device_name(i)}")

# DICOM decoders. Compressed transfer syntaxes need at least one of these.
DECODERS = {}
for mod in ("pylibjpeg", "libjpeg", "openjpeg", "gdcm"):
    try:
        __import__(mod)
        DECODERS[mod] = True
    except Exception:
        DECODERS[mod] = False
print(f"decoders    : {DECODERS}")
if not (DECODERS["gdcm"] or (DECODERS["pylibjpeg"] and DECODERS["libjpeg"])):
    print("  !! No JPEG decoder. Compressed series will be skipped. Add "
          "pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg python-gdcm to the "
          "notebook dependency list and restart.")

COMP_ROOT = resolve_comp_root()
TRAIN_CSV = f"{COMP_ROOT}/train.csv"
SERIES_ROOT = f"{COMP_ROOT}/train_series"
print(f"comp root   : {COMP_ROOT}")
print(f"train csv   : {TRAIN_CSV}  exists={Path(TRAIN_CSV).exists()}")
print(f"series root : {SERIES_ROOT}  exists={Path(SERIES_ROOT).exists()}")

train_df = pd.read_csv(TRAIN_CSV)
print(f"\ntrain.csv   : {train_df.shape}  columns={list(train_df.columns)}")
print(train_df.head(3))

gpu_mem("before model load")

PREFLIGHT
python      : 3.12.13
torch       : 2.10.0+cu128  (cuda 12.8)
torchvision : 0.25.0+cu128
pydicom     : 3.0.2
cuda avail  : True  n_gpu=2
  cuda:0 -> Tesla T4
  cuda:1 -> Tesla T4
decoders    : {'pylibjpeg': False, 'libjpeg': False, 'openjpeg': False, 'gdcm': False}
  !! No JPEG decoder. Compressed series will be skipped. Add pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg python-gdcm to the notebook dependency list and restart.
comp root   : /kaggle/input/competitions/rsna-knee-abnormality-detection
train csv   : /kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv  exists=True
series root : /kaggle/input/competitions/rsna-knee-abnormality-detection/train_series  exists=True

train.csv   : (4407, 14)  columns=['StudyInstanceUID', 'Report', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
                                    StudyInstanceUID  \
0  1.2.826.0.1.3680

## 2. Pretrained backbone as a frozen feature extractor

Offline-first weight resolution: attached RadImageNet PyTorch port → staged ImageNet weights → Kaggle Models mount → (dev sessions only) torchvision download. Refuses to run on a random init.


In [5]:
# =============================================================================
# CELL 2 - Pretrained backbone as a frozen feature extractor
# =============================================================================
# WEIGHTS, and how to get them into an offline run. Three routes, in order of
# preference:
#
#   (A) RadImageNet ResNet50, PyTorch port.  RadImageNet was trained on 1.35M
#       CT/MR/US slices, so its filters transfer to knee MR far better than
#       ImageNet's. The OFFICIAL release (github.com/BMEII-AI/RadImageNet) is
#       Keras .h5 - we cannot load that here. Attach a PyTorch port as a Kaggle
#       Dataset (search Kaggle Datasets for "RadImageNet pytorch"), or convert
#       the .h5 yourself in a dev session. The loader below scans /kaggle/input
#       for *.pt/*.pth/*.safetensors whose path mentions radimagenet.
#
#   (B) ImageNet ResNet50 staged offline. In a dev session WITH internet on,
#       run  stage_weights_for_offline()  -> writes resnet50_imagenet.pth into
#       /kaggle/working, then "Save Version" and attach that output as a Dataset
#       to the offline run. The loader finds it automatically.
#
#   (C) Kaggle Models mount, e.g. /kaggle/input/resnet50/pytorch/v1/1/*.pth.
#       Also picked up by the scan.
#
# NOT used, and why: MedicalNet is 3D ResNets for volumetric CT/MR segmentation
# (different input rank, different stem) and torchxrayvision is chest-radiograph
# DenseNet121 - neither is a drop-in 2D knee-MR encoder, so ImageNet is the
# better fallback than either.
#
# FALLBACK NOTE: if no RadImageNet weights are found we fall back to standard
# ImageNet-pretrained torchvision ResNet50. That is a real downgrade in domain
# match but still a strong generic edge/texture encoder. The weight source is
# printed and recorded in WEIGHT_SOURCE / stamped into the saved checkpoint.
# -----------------------------------------------------------------------------

import re
from collections import OrderedDict

ALLOW_RANDOM_INIT = False   # guard: never burn an hour extracting noise
FEATURE_DIM = 2048          # ResNet50 penultimate width

# Normalisation presets. ImageNet ports expect ImageNet mean/std; the common
# RadImageNet ports were trained on plain [0,1] rescaled images. If the port you
# attach documents ImageNet stats, set RADIMAGENET_USES_IMAGENET_NORM = True.
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
RADIMAGENET_USES_IMAGENET_NORM = False


def find_local_weight_files():
    """Every plausible checkpoint sitting in an attached dataset / model / cache."""
    roots = [Path("/kaggle/input"), WORK, Path.home() / ".cache/torch/hub/checkpoints"]
    exts = ("*.pth", "*.pt", "*.safetensors", "*.bin")
    hits = []
    for root in roots:
        if not root.exists():
            continue
        for ext in exts:
            # depth-limited glob: /kaggle/input can be enormous
            for pat in (ext, f"*/{ext}", f"*/*/{ext}", f"*/*/*/{ext}", f"*/*/*/*/{ext}"):
                hits.extend(root.glob(pat))
    # de-dup, drop our own outputs
    out, seen = [], set()
    for h in hits:
        s = str(h)
        if s in seen or "slice_features" in s:
            continue
        seen.add(s)
        out.append(h)
    return out


def score_candidate(path):
    """Rank checkpoints: RadImageNet ResNet50 > any ResNet50 > anything else."""
    s = str(path).lower()
    score = 0
    if "radimagenet" in s or "rad_imagenet" in s:
        score += 100
    if "resnet50" in s or "resnet_50" in s or "r50" in s:
        score += 10
    if "imagenet" in s:
        score += 5
    if "convnext" in s:
        score += 3
    return score


def load_state_dict_any(path):
    """torch .pth/.pt/.bin or .safetensors -> plain dict of tensors."""
    path = str(path)
    if path.endswith(".safetensors"):
        from safetensors.torch import load_file
        return load_file(path)
    obj = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(obj, dict):
        for key in ("state_dict", "model_state_dict", "model", "net"):
            if key in obj and isinstance(obj[key], dict):
                obj = obj[key]
                break
    if isinstance(obj, nn.Module):          # someone pickled the whole model
        obj = obj.state_dict()
    return obj


def remap_keys(sd):
    """Strip the wrapper prefixes ports pick up (DataParallel, Lightning, etc.)."""
    out = OrderedDict()
    for k, v in sd.items():
        nk = k
        for pref in ("module.", "backbone.", "encoder.", "model.", "features.",
                     "resnet.", "net."):
            if nk.startswith(pref):
                nk = nk[len(pref):]
        out[nk] = v
    return out


def stage_weights_for_offline(dest=WORK / "resnet50_imagenet.pth"):
    """Run ONCE in a dev session with internet ON, then attach the output as a
    Dataset so the offline competition run can find these weights."""
    w = torchvision.models.ResNet50_Weights.IMAGENET1K_V2   # raises offline
    m = torchvision.models.resnet50(weights=w)
    torch.save(m.state_dict(), dest)
    print(f"Staged ImageNet ResNet50 -> {dest} ({dest.stat().st_size/1e6:.0f} MB)")
    print("Now: Save Version -> attach this notebook's output as a Dataset to the offline run.")
    return dest


def build_backbone():
    """Return (feature_extractor, feature_dim, (mean, std), source_string).

    The extractor maps [B,3,224,224] -> [B,2048]: full ResNet50 with the final
    fc replaced by Identity, so what comes out is the global-average-pooled
    conv feature - exactly what we want for downstream MIL/attention pooling.
    """
    model = torchvision.models.resnet50(weights=None)
    source, norm = None, (IMAGENET_MEAN, IMAGENET_STD)

    candidates = sorted(find_local_weight_files(), key=score_candidate, reverse=True)
    print(f"Found {len(candidates)} candidate checkpoint(s); top 5:")
    for c in candidates[:5]:
        print(f"   [{score_candidate(c):3d}] {c}")

    for cand in candidates:
        if score_candidate(cand) < 10:      # not a resnet50 - skip
            continue
        try:
            sd = remap_keys(load_state_dict_any(cand))
            missing, unexpected = model.load_state_dict(sd, strict=False)
            # fc.* is expected to be missing/unexpected - it is a headless port
            hard_missing = [k for k in missing if not k.startswith("fc.")]
            loaded = len(model.state_dict()) - len(hard_missing)
            if hard_missing and len(hard_missing) > 10:
                print(f"   x {cand.name}: {len(hard_missing)} conv/bn keys unmatched - skipping")
                continue
            is_rad = "radimagenet" in str(cand).lower()
            source = ("RadImageNet-ResNet50" if is_rad else "ImageNet-ResNet50") + f" <- {cand}"
            if is_rad and not RADIMAGENET_USES_IMAGENET_NORM:
                norm = ((0.0, 0.0, 0.0), (1.0, 1.0, 1.0))   # plain [0,1] inputs
            print(f"   OK loaded {loaded}/{len(model.state_dict())} tensors from {cand.name}")
            if unexpected:
                print(f"      ({len(unexpected)} unexpected keys ignored, e.g. {unexpected[:3]})")
            break
        except Exception as e:
            print(f"   x {cand.name}: {type(e).__name__}: {e}")

    if source is None:
        # Nothing attached. torchvision checks the torch hub cache first and only
        # then hits the network, so this works in a dev session AND in an offline
        # session that happens to have the weights cached. Offline with an empty
        # cache it raises, and the guard below turns that into a clear message.
        # FALLBACK: plain ImageNet, not RadImageNet.
        try:
            model = torchvision.models.resnet50(
                weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
            source = "ImageNet-ResNet50 <- torchvision cache/download (FALLBACK, not RadImageNet)"
        except Exception as e:
            print(f"   x torchvision IMAGENET1K_V2 unavailable: {type(e).__name__}: {e}")

    if source is None:
        msg = ("No pretrained weights available offline. Features from a randomly "
               "initialised backbone are worthless - see this cell's header for the "
               "three ways to stage weights. Set ALLOW_RANDOM_INIT=True to override.")
        if not ALLOW_RANDOM_INIT:
            raise RuntimeError(msg)
        print("!! " + msg)
        source = "RANDOM-INIT (no pretraining!)"

    model.fc = nn.Identity()        # 2048-d feature vector instead of 1000 logits
    return model, FEATURE_DIM, norm, source


# --------------------------------------------------------------------------- #
# Build, move to GPU(s), report memory
# --------------------------------------------------------------------------- #

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

gpu_mem("before backbone")
backbone, FEATURE_DIM, (NORM_MEAN, NORM_STD), WEIGHT_SOURCE = build_backbone()
backbone = backbone.to(DEVICE).eval()
for p in backbone.parameters():                 # frozen: pure feature extraction
    p.requires_grad_(False)

# DataParallel splits each batch across both T4s -> roughly 2x throughput, as
# long as the DataLoader can feed it (DICOM decode is the real bottleneck).
MULTI_GPU = torch.cuda.device_count() > 1
feature_model = nn.DataParallel(backbone) if MULTI_GPU else backbone

gpu_mem("after backbone")
print(f"\nweight source : {WEIGHT_SOURCE}")
print(f"feature dim   : {FEATURE_DIM}")
print(f"normalisation : mean={NORM_MEAN} std={NORM_STD}")
print(f"device        : {DEVICE}  DataParallel={MULTI_GPU}")

# Shape sanity check - catches a wrong head or a bad load immediately.
with torch.no_grad(), torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
    dummy = torch.randn(4, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    out = feature_model(dummy)
print(f"sanity check  : {tuple(dummy.shape)} -> {tuple(out.shape)} (expect [4, {FEATURE_DIM}])")
assert out.shape == (4, FEATURE_DIM), "unexpected feature shape"
del dummy, out
torch.cuda.empty_cache()

[gpu_mem] before backbone: cuda:0 0.02/5.24 of 14.6 GiB | cuda:1 0.02/5.24 of 14.6 GiB
Found 1 candidate checkpoint(s); top 5:
   [115] /kaggle/input/datasets/shigechan/radimagenet-resnet50-official/ResNet50.pt
   x ResNet50.pt: 265 conv/bn keys unmatched - skipping
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 197MB/s] 
/tmp/ipykernel_58/966569008.py:215: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


[gpu_mem] after backbone: cuda:0 0.11/5.24 of 14.6 GiB | cuda:1 0.02/5.24 of 14.6 GiB

weight source : ImageNet-ResNet50 <- torchvision cache/download (FALLBACK, not RadImageNet)
feature dim   : 2048
normalisation : mean=(0.485, 0.456, 0.406) std=(0.229, 0.224, 0.225)
device        : cuda  DataParallel=True
sanity check  : (4, 3, 224, 224) -> (4, 2048) (expect [4, 2048])
